# Chapter 13 — Build It Yourself: Quantization

Implement the quantize/dequantize map, compare int8 vs int4 error, see why **per-channel** scaling
beats per-tensor on outliers, then quantize a real GPT and watch it get smaller with barely any
change to its output.

> 💡 **No GPU needed** — it's arithmetic on weight tensors. Running a ✍️ cell before you fill it
> prints a friendly ⚠️, not an error.

In [ ]:
import torch
import torch.nn as nn
print('ready')

## Step 1 — Implement quantize/dequantize  ✍️ Your turn

Symmetric int8: pick a **scale** (`max|w| / 127`), divide-and-round to integers, multiply back.
Fill the three lines and watch the recovered floats land near the originals.

<details><summary>Stuck? Show the answer</summary>

```python
scale = w.abs().max() / qmax
q = torch.clamp(torch.round(w / scale), -128, 127)
w_hat = q * scale
```
</details>

In [ ]:
w = torch.tensor([0.10, -0.42, 0.98, -0.05, 0.61])
qmax = 127
# ✍️ symmetric int8 quantize -> dequantize
scale = None      # replace
q = None          # replace
w_hat = None      # replace

print('integers :', None if q is None else q.int().tolist(), '  <- stored, 1 byte each')
print('recovered:', None if w_hat is None else [round(x, 4) for x in w_hat.tolist()])

In [ ]:
# ▶️ Check your work
try:
    if scale is None or q is None or w_hat is None:
        print('⚠️  Fill in the ✍️ cell above (scale, q, w_hat), then re-run. (Expected until you do.)')
    elif q.int().tolist() == [13, -54, 127, -6, 79] and torch.allclose(w, w_hat, atol=scale.item()):
        print('✅ Correct — weights quantized to integers and recovered within a rounding step.')
    else:
        print('⚠️  Not quite — check scale = max|w|/127, round, clamp to [-128,127], then × scale.')
except Exception as e:
    print('❌', e)

## Step 2 — int8 vs int4: fewer bits, more error  ▶️ Run this

int8 has 256 integer levels, int4 only 16 — a 16× coarser grid. Run it on a random weight matrix
and see the error jump when you drop to int4.

In [ ]:
def quant_dequant(w, bits, per_channel=False):
    qmax = 2**(bits-1) - 1
    scale = w.abs().amax(1, keepdim=True)/qmax if per_channel else w.abs().max()/qmax
    return torch.clamp(torch.round(w/scale), -qmax-1, qmax) * scale

torch.manual_seed(0); w = torch.randn(64, 64)
for bits in [8, 4]:
    err = (w - quant_dequant(w, bits)).pow(2).mean().sqrt() / w.std()
    print(f'int{bits}: relative error {err:.4f}')

## Step 3 — Per-channel beats per-tensor  ✍️ Your turn

With outlier rows (values 10–20× the rest), one *per-tensor* scale crushes the ordinary weights.
**Per-channel** — one scale per row — fixes it. Fill the per-channel scale and compare the error.

<details><summary>Stuck? Show the answer</summary>

```python
scale_pc = w.abs().amax(dim=1, keepdim=True) / qmax     # max per ROW (dim=1)
```
</details>

In [ ]:
torch.manual_seed(0); w = torch.randn(64, 64)
w[7] *= 20; w[30] *= 15                       # a couple of outlier rows
qmax = 7                                      # int4

scale_pt = w.abs().max() / qmax               # per-tensor: one scale for everything
wh_pt = torch.clamp(torch.round(w/scale_pt), -8, 7) * scale_pt

# ✍️ per-channel: one scale per ROW (dim=1, keepdim=True)
scale_pc = None      # replace
wh_pc = None if scale_pc is None else torch.clamp(torch.round(w/scale_pc), -8, 7) * scale_pc

err = lambda wh: ((w - wh).pow(2).mean().sqrt() / w.std()).item()
print('per-tensor  int4 error:', round(err(wh_pt), 4))
print('per-channel int4 error:', None if wh_pc is None else round(err(wh_pc), 4))

In [ ]:
# ▶️ Check your work
try:
    if scale_pc is None or wh_pc is None:
        print('⚠️  Fill in the ✍️ per-channel scale (amax over dim=1), then re-run. (Expected until you do.)')
    elif err(wh_pc) < 0.15:      # correct per-ROW (dim=1) lands ~0.12; a per-column (dim=0) fill gives ~0.21 and fails
        print(f'✅ Per-channel error {err(wh_pc):.3f} << per-tensor {err(wh_pt):.3f} — outliers no longer crush the rest.')
    else:
        print(f'⚠️  Error {err(wh_pc):.3f} is too high — the scale must be per ROW (dim=1, keepdim=True).')
except Exception as e:
    print('❌', e)

## Step 4 — Quantize a real GPT  ▶️ Run this

Now the payoff: quantize every `nn.Linear` weight of a small GPT (per-channel) to int8, then int4,
and measure how far the output logits **drift** from fp32 — plus the size drop. (Just run the
cells; the model is defined for you.)

In [ ]:
import math, copy
V, C, NH, NL, BLK = 65, 128, 4, 3, 64; HD = C // NH
class Attn(nn.Module):
    def __init__(s): super().__init__(); s.q,s.k,s.v,s.proj=(nn.Linear(C,C) for _ in range(4))
    def forward(s,x):
        B,T,_=x.shape; q,k,v=(l(x).view(B,T,NH,HD).transpose(1,2) for l in (s.q,s.k,s.v))
        m=torch.triu(torch.ones(T,T)*float('-inf'),1)
        a=torch.softmax((q@k.transpose(-2,-1))/math.sqrt(HD)+m,-1)@v
        return s.proj(a.transpose(1,2).reshape(B,T,C))
class Blk(nn.Module):
    def __init__(s): super().__init__(); s.l1,s.l2=nn.LayerNorm(C),nn.LayerNorm(C); s.at=Attn(); s.ff=nn.Sequential(nn.Linear(C,4*C),nn.GELU(),nn.Linear(4*C,C))
    def forward(s,x): x=x+s.at(s.l1(x)); return x+s.ff(s.l2(x))
class GPT(nn.Module):
    def __init__(s): super().__init__(); s.tok=nn.Embedding(V,C); s.pos=nn.Embedding(BLK,C); s.bl=nn.ModuleList([Blk() for _ in range(NL)]); s.lf=nn.LayerNorm(C); s.hd=nn.Linear(C,V)
    def forward(s,idx):
        T=idx.size(1); x=s.tok(idx)+s.pos(torch.arange(T))
        for b in s.bl: x=b(x)
        return s.hd(s.lf(x))

In [ ]:
torch.manual_seed(0); model = GPT().eval()
idx = torch.randint(0, V, (4, BLK), generator=torch.Generator().manual_seed(1))
with torch.no_grad(): ref = model(idx)
n = sum(p.numel() for p in model.parameters())

print(f'{"format":>7} {"size":>9} {"logit drift":>13}')
print(f'{"fp16":>7} {n*2/1e6:>6.2f} MB {"0.0000 (ref)":>13}')
for bits in [8, 4]:
    mq = copy.deepcopy(model); qmax = 2**(bits-1) - 1
    for mod in mq.modules():
        if isinstance(mod, nn.Linear):
            w = mod.weight.data; s = w.abs().amax(1, keepdim=True)/qmax
            mod.weight.data = torch.clamp(torch.round(w/s), -qmax-1, qmax) * s
    with torch.no_grad(): out = mq(idx)
    drift = ((out-ref).pow(2).mean().sqrt() / ref.std()).item()
    print(f'{"int"+str(bits):>7} {n*bits/8/1e6:>6.2f} MB {drift:>13.4f}')

## 🎉 You quantized a model.

int8 halves the model with a ~0.4% output drift (you couldn't tell its stories apart); int4
quarters it for a few % drift. This is how the models you download to run locally are stored.
Take it to your **GPT** in the [mini-project](../project/), then on to
[Chapter 14 — Finetuning](../../14-finetuning-sft/). 🗜️